In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mitigasi ralat rangkaian tensor (TEM): Fungsi Qiskit oleh Algorithmiq
*Lihat [rujukan API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Qiskit Functions adalah ciri eksperimental yang hanya tersedia untuk pengguna Plan Premium IBM Quantum&reg;, Plan Flex, dan Plan On-Prem (melalui IBM Quantum Platform API). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.


<Accordion>
<AccordionItem title="Versi pakej">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## Gambaran keseluruhan
Kaedah Mitigasi Ralat Rangkaian Tensor (TEM) Algorithmiq adalah algoritma
kuantum-klasikal hibrid yang direka untuk melakukan mitigasi hingar sepenuhnya
pada peringkat pasca-pemprosesan klasikal. Dengan TEM, pengguna boleh mengira
nilai jangkaan pemerhatian yang mengurangkan ralat yang disebabkan oleh hingar
yang tidak dapat dielakkan pada perkakasan kuantum dengan ketepatan dan
kecekapan kos yang ditingkatkan, menjadikannya pilihan yang sangat menarik bagi
penyelidik kuantum dan pengamal industri.

Kaedah ini terdiri daripada membina rangkaian tensor yang mewakili songsangan
saluran hingar global yang mempengaruhi keadaan pemproses kuantum dan kemudian
menggunakan peta tersebut pada hasil pengukuran lengkap maklumat yang diperoleh
daripada keadaan bising untuk mendapatkan penganggar tidak berat sebelah bagi
pemerhatian.

Sebagai kelebihan, TEM memanfaatkan pengukuran lengkap maklumat untuk memberi
akses kepada set nilai jangkaan pemerhatian yang dikurangkan yang luas dan
mempunyai overhead persampelan yang optimum pada perkakasan kuantum, seperti
yang diterangkan dalam Filippov et al. (2023),
[arXiv:2307.11740](https://arxiv.org/abs/2307.11740), dan Filippov et al.
(2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Overhead
pengukuran merujuk kepada bilangan pengukuran tambahan yang diperlukan untuk
melakukan mitigasi ralat yang cekap, faktor kritikal dalam kebolehlaksanaan
pengiraan kuantum. Oleh itu, TEM berpotensi membolehkan kelebihan kuantum dalam
senario kompleks, seperti aplikasi dalam bidang huru-hara kuantum, fizik banyak
badan, dinamik Hubbard, dan simulasi kimia molekul kecil.

Ciri-ciri dan manfaat utama TEM boleh diringkaskan sebagai:

1.  **Overhead pengukuran optimum**: TEM adalah optimum berkenaan
[had teori](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
bermakna tiada kaedah yang boleh mencapai overhead pengukuran yang lebih kecil.
Dalam erti kata lain, TEM memerlukan bilangan minimum pengukuran tambahan untuk
melakukan mitigasi ralat. Ini seterusnya bermakna TEM menggunakan masa runtime
kuantum yang minimum.
2.  **Kos efektif**: Memandangkan TEM mengendalikan mitigasi hingar sepenuhnya
dalam peringkat pasca-pemprosesan, tidak perlu menambah litar tambahan kepada
komputer kuantum, yang bukan sahaja menjadikan pengiraan lebih murah tetapi
juga mengurangkan risiko memperkenalkan ralat tambahan akibat ketidaksempurnaan
peranti kuantum.
3.  **Anggaran pelbagai pemerhatian**: Berkat pengukuran lengkap maklumat, TEM
menganggarkan pelbagai pemerhatian dengan cekap menggunakan data pengukuran
yang sama daripada komputer kuantum.
4.  **Mitigasi ralat pengukuran**: Fungsi TEM Qiskit juga termasuk
[kaedah mitigasi ralat pengukuran proprietari](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
yang mampu mengurangkan ralat bacaan dengan ketara selepas jalankan kalibrasi
yang singkat.
5.  **Ketepatan**: TEM meningkatkan ketepatan dan kebolehpercayaan simulasi
kuantum digital dengan ketara, menjadikan algoritma kuantum lebih tepat dan
boleh dipercayai.
## Keterangan
Fungsi TEM membolehkan anda mendapatkan nilai jangkaan yang dikurangkan ralat
untuk pelbagai pemerhatian pada Circuit kuantum dengan overhead persampelan
yang minimum. Circuit diukur dengan ukuran operator bernilai pengendali positif
yang lengkap maklumat (IC-POVM), dan hasil pengukuran yang dikumpulkan diproses
pada komputer klasikal. Pengukuran ini digunakan untuk melakukan kaedah
rangkaian tensor dan membina peta songsangan hingar. Fungsi ini menggunakan
peta yang menyongsangkan sepenuhnya keseluruhan Circuit bising menggunakan
rangkaian tensor untuk mewakili lapisan bising.

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Anggaran pemerhatian O yang dikurangkan ralat melalui hasil pengukuran pasca-pemprosesan pemproses kuantum bising. U dan N menandakan operasi kuantum ideal dan peta hingar yang berkaitan, yang secara amnya boleh bukan tempatan (dan diperluas kepada kotak kelabu). D mewakili tensor pengendali yang dual kepada kesan dalam pengukuran IC. Modul mitigasi hingar M adalah rangkaian tensor yang dikontrak dengan cekap dari tengah ke luar. Iterasi pertama penguncupan diwakili oleh garis ungu bertitik, yang kedua oleh garis putus-putus, dan yang ketiga oleh garis pepejal.")

Sebaik sahaja litar diserahkan ke fungsi, ia ditranspilasikan dan dioptimumkan
untuk meminimumkan bilangan lapisan dengan get dua qubit (get yang lebih bising
pada peranti kuantum). Hingar yang mempengaruhi lapisan dipelajari melalui
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
menggunakan model hingar Pauli-Lindblad jarang seperti yang diterangkan dalam
E. van den Berg, Z. Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model hingar adalah keterangan tepat hingar pada peranti yang mampu menangkap
ciri-ciri halus, termasuk cross-talk qubit. Walau bagaimanapun, hingar pada
peranti boleh turun naik dan hanyut dan hingar yang dipelajari mungkin tidak
tepat pada ketika anggaran dibuat. Ini boleh mengakibatkan keputusan yang tidak
tepat.
## Mulakan
Sahkan menggunakan [kunci API IBM Quantum Platform](http://quantum.cloud.ibm.com/)
anda, dan pilih fungsi TEM seperti berikut. (Coretan ini mengandaikan anda sudah
[menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client)
ke persekitaran tempatan anda.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load your function
tem = catalog.load(tem_function_name)

Gunakan API Qiskit Serverless untuk memeriksa status beban kerja Fungsi Qiskit anda:

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Use the Qiskit Serverless APIs to check your Qiskit Function workload's status:

In [3]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())

30db31ed-d252-4ca8-a5f0-5eb9b5075a4c
QUEUED


Anda boleh mengembalikan keputusan seperti:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.09417719217134088


<Admonition type="info">
  The expected value for the noiseless circuit for the given operator should be around `0.18409094298943401`.
</Admonition>

## Get support

Reach out to [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi)

Be sure to include the following information:
- Qiskit Function Job ID (`qiskit-ibm-catalog`), `job.job_id`
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
- Visit the [API reference](/docs/api/functions/algorithmiq-tem) for this Qiskit Function.

</Admonition>